In [ ]:
"""
================================================================================
COMPLETE PERSON RE-IDENTIFICATION PIPELINE
Market-1501 Dataset | OSNet + GaitSet + FaceNet
================================================================================
"""

import os
import glob
import re
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.notebook import tqdm

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ================================================================================
# PART 1: DATA LOADING
# ================================================================================

print("\n" + "="*80)
print("PART 1: LOADING MARKET-1501 DATASET")
print("="*80)

# Download dataset
import kagglehub
path = kagglehub.dataset_download("pengcw1/market-1501")
market_root = f'{path}/Market-1501-v15.09.15'
print(f"Dataset path: {market_root}")

class Market1501Dataset(Dataset):
    """Market-1501 Dataset for Person Re-Identification"""
    
    def __init__(self, root_dir, mode='train', transform=None):
        self.root_dir = root_dir
        self.mode = mode
        self.transform = transform
        
        # Determine folder based on mode
        if self.mode == 'train':
            self.sub_path = 'bounding_box_train'
        elif self.mode == 'gallery':
            self.sub_path = 'bounding_box_test'
        elif self.mode == 'query':
            self.sub_path = 'query'
        
        self.data_path = os.path.join(self.root_dir, self.sub_path)
        self.data = []
        self.id_to_images = {}
        
        self._parse_data()
        print(f"Loaded {self.mode}: {len(self.data)} images, {len(self.id_to_images)} identities")
    
    def _parse_data(self):
        pattern = re.compile(r'([-\d]+)_c(\d)')
        img_paths = glob.glob(os.path.join(self.data_path, '*.jpg'))
        
        for img_path in img_paths:
            filename = os.path.basename(img_path)
            
            # Skip junk images
            if filename.startswith('-1') or filename.startswith('0000'):
                continue
            
            pid, camid = map(int, pattern.search(filename).groups())
            self.data.append((img_path, pid, camid))
            
            # For training: organize by PID
            if self.mode == 'train':
                if pid not in self.id_to_images:
                    self.id_to_images[pid] = []
                self.id_to_images[pid].append(img_path)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        if self.mode == 'train':
            # Return triplet: anchor, positive, negative
            anchor_path, anchor_pid, _ = self.data[idx]
            
            # Positive: same PID
            pos_path = random.choice(self.id_to_images[anchor_pid])
            
            # Negative: different PID
            all_pids = list(self.id_to_images.keys())
            neg_pid = random.choice(all_pids)
            while neg_pid == anchor_pid:
                neg_pid = random.choice(all_pids)
            neg_path = random.choice(self.id_to_images[neg_pid])
            
            # Load images
            anchor = Image.open(anchor_path).convert('RGB')
            positive = Image.open(pos_path).convert('RGB')
            negative = Image.open(neg_path).convert('RGB')
            
            if self.transform:
                anchor = self.transform(anchor)
                positive = self.transform(positive)
                negative = self.transform(negative)
            
            return anchor, positive, negative, torch.tensor(anchor_pid)
        else:
            # Return single image for evaluation
            img_path, pid, camid = self.data[idx]
            img = Image.open(img_path).convert('RGB')
            
            if self.transform:
                img = self.transform(img)
            
            return img, torch.tensor(pid), torch.tensor(camid)

# Define transforms
transform_train = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = Market1501Dataset(market_root, mode='train', transform=transform_train)
query_dataset = Market1501Dataset(market_root, mode='query', transform=transform_test)
gallery_dataset = Market1501Dataset(market_root, mode='gallery', transform=transform_test)

# Create train/val split
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(
    train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Query: {len(query_dataset)} | Gallery: {len(gallery_dataset)}")

# ================================================================================
# PART 2: MODEL DEFINITIONS
# ================================================================================

print("\n" + "="*80)
print("PART 2: DEFINING MODELS")
print("="*80)

# -------------------- OSNet Model --------------------
print("\nInstalling torchreid...")
import subprocess
subprocess.run(["pip", "install", "-q", "torchreid"], check=True)

import torchreid

num_train_ids = len(train_dataset.id_to_images)
print(f"Number of training identities: {num_train_ids}")

def build_osnet():
    model = torchreid.models.build_model(
        name='osnet_x1_0',
        num_classes=num_train_ids,
        loss='triplet'
    )
    return model.to(device)

# -------------------- GaitSet Model --------------------
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, **kwargs):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        return F.leaky_relu(self.bn(self.conv(x)), 0.1, inplace=True)

class SetBlock(nn.Module):
    def __init__(self, forward_block, pooling=False):
        super(SetBlock, self).__init__()
        self.forward_block = forward_block
        self.pooling = pooling
        if pooling:
            self.pool2d = nn.MaxPool2d(2)
    
    def forward(self, x):
        n, s, c, h, w = x.size()
        x = self.forward_block(x.view(-1, c, h, w))
        if self.pooling:
            x = self.pool2d(x)
        _, c, h, w = x.size()
        return x.view(n, s, c, h, w)

class GaitSet(nn.Module):
    def __init__(self, num_classes=751):
        super(GaitSet, self).__init__()
        
        self.layer1 = SetBlock(BasicConv2d(3, 32, 3, padding=1), True)
        self.layer2 = SetBlock(BasicConv2d(32, 64, 3, padding=1), True)
        self.layer3 = SetBlock(BasicConv2d(64, 128, 3, padding=1), True)
        self.layer4 = SetBlock(BasicConv2d(128, 256, 3, padding=1))
        
        self.gl_conv = BasicConv2d(256, 256, 3, padding=1)
        self.classifier = nn.Linear(256, num_classes)
    
    def frame_max(self, x):
        return torch.max(x, 1)[0]
    
    def forward(self, x):
        # Handle input: [B, C, H, W] -> [B, 1, C, H, W]
        if x.dim() == 4:
            x = x.unsqueeze(1)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.frame_max(x)
        x = self.gl_conv(x)
        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(x.size(0), -1)
        
        if self.training:
            logits = self.classifier(x)
            return x, logits
        return x

def build_gaitset():
    return GaitSet(num_classes=num_train_ids).to(device)

# -------------------- FaceNet Model --------------------
print("\nInstalling facenet-pytorch...")
subprocess.run(["pip", "install", "-q", "facenet-pytorch"], check=True)

from facenet_pytorch import InceptionResnetV1

def build_facenet():
    model = InceptionResnetV1(pretrained='vggface2')
    return model.to(device)

print("✓ All models defined successfully")

# ================================================================================
# PART 3: TRAINING UTILITIES
# ================================================================================

print("\n" + "="*80)
print("PART 3: TRAINING UTILITIES")
print("="*80)

class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super(TripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        pos_dist = (anchor - positive).pow(2).sum(1)
        neg_dist = (anchor - negative).pow(2).sum(1)
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()

def train_model(model, train_loader, val_loader, model_name, 
                num_epochs=15, lr=0.0003, weight_decay=5e-4):
    """Generic training function for all models"""
    
    print(f"\n{'='*60}")
    print(f"TRAINING {model_name.upper()}")
    print(f"{'='*60}")
    
    triplet_loss = TripletLoss(margin=0.3)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_losses = []
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for anchor, positive, negative, _ in loop:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            anchor_out = model(anchor)
            positive_out = model(positive)
            negative_out = model(negative)
            
            # Extract features (handle different output formats)
            anchor_feat = anchor_out[0] if isinstance(anchor_out, tuple) else anchor_out
            positive_feat = positive_out[0] if isinstance(positive_out, tuple) else positive_out
            negative_feat = negative_out[0] if isinstance(negative_out, tuple) else negative_out
            
            # Normalize
            anchor_feat = F.normalize(anchor_feat, p=2, dim=1)
            positive_feat = F.normalize(positive_feat, p=2, dim=1)
            negative_feat = F.normalize(negative_feat, p=2, dim=1)
            
            # Loss
            loss = triplet_loss(anchor_feat, positive_feat, negative_feat)
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
            loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
        
        avg_train_loss = np.mean(train_losses)
        history['train_loss'].append(avg_train_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # Validation
        model.eval()
        val_losses = []
        
        with torch.no_grad():
            for anchor, positive, negative, _ in val_loader:
                anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
                
                anchor_out = model(anchor)
                positive_out = model(positive)
                negative_out = model(negative)
                
                anchor_feat = anchor_out[0] if isinstance(anchor_out, tuple) else anchor_out
                positive_feat = positive_out[0] if isinstance(positive_out, tuple) else positive_out
                negative_feat = negative_out[0] if isinstance(negative_out, tuple) else negative_out
                
                anchor_feat = F.normalize(anchor_feat, p=2, dim=1)
                positive_feat = F.normalize(positive_feat, p=2, dim=1)
                negative_feat = F.normalize(negative_feat, p=2, dim=1)
                
                loss = triplet_loss(anchor_feat, positive_feat, negative_feat)
                val_losses.append(loss.item())
        
        avg_val_loss = np.mean(val_losses)
        history['val_loss'].append(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f'best_{model_name.lower()}_model.pth')
            print(f"  ✓ Saved best model (val_loss: {avg_val_loss:.4f})")
        
        scheduler.step()
    
    print(f"\n✓ {model_name} training completed!")
    return model, history

# ================================================================================
# PART 4: EVALUATION UTILITIES
# ================================================================================

print("\n" + "="*80)
print("PART 4: EVALUATION UTILITIES")
print("="*80)

def evaluate_model(model, query_dataset, gallery_dataset, model_name, batch_size=64):
    """Comprehensive evaluation"""
    
    print(f"\n{'='*60}")
    print(f"EVALUATING {model_name.upper()}")
    print(f"{'='*60}")
    
    model.eval()
    start_time = time.time()
    
    def extract_features(dataset, desc):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        features, pids, camids = [], [], []
        
        with torch.no_grad():
            for img, pid, camid in tqdm(loader, desc=desc):
                img = img.to(device)
                output = model(img)
                
                feat = output[0] if isinstance(output, (tuple, list)) else output
                feat = F.normalize(feat, p=2, dim=1)
                
                features.append(feat.cpu())
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
        
        return torch.cat(features, 0), np.array(pids), np.array(camids)
    
    # Extract features
    q_feats, q_pids, q_camids = extract_features(query_dataset, f"{model_name} Query")
    g_feats, g_pids, g_camids = extract_features(gallery_dataset, f"{model_name} Gallery")
    
    extraction_time = time.time() - start_time
    
    # Compute distances
    print("Computing distance matrix...")
    m, n = q_feats.size(0), g_feats.size(0)
    dist_mat = torch.pow(q_feats, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    # Calculate metrics
    print("Calculating metrics...")
    max_rank = 50
    cmc_scores = np.zeros(max_rank)
    ap_scores = []
    
    for i in range(len(q_pids)):
        q_pid, q_cam = q_pids[i], q_camids[i]
        q_dist = dist_mat[i]
        
        sorted_indices = np.argsort(q_dist)
        sorted_pids = g_pids[sorted_indices]
        sorted_cams = g_camids[sorted_indices]
        
        junk_mask = (sorted_pids == q_pid) & (sorted_cams == q_cam)
        keep_mask = ~junk_mask
        matches = (sorted_pids[keep_mask] == q_pid).astype(np.int32)
        
        if len(matches) == 0:
            continue
        
        # CMC
        raw_cmc = matches.cumsum()
        raw_cmc[raw_cmc > 1] = 1
        cmc_padded = np.ones(max_rank) * raw_cmc[-1] if len(raw_cmc) > 0 else np.zeros(max_rank)
        cmc_padded[:min(len(raw_cmc), max_rank)] = raw_cmc[:max_rank]
        cmc_scores += cmc_padded
        
        # AP
        num_rel = matches.sum()
        if num_rel > 0:
            tmp_cmc = matches.cumsum()
            tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
            ap = np.sum(np.array(tmp_cmc) * matches) / num_rel
            ap_scores.append(ap)
    
    cmc_scores /= len(q_pids)
    
    results = {
        'model_name': model_name,
        'mAP': np.mean(ap_scores) * 100,
        'rank1': cmc_scores[0] * 100,
        'rank5': cmc_scores[4] * 100,
        'rank10': cmc_scores[9] * 100,
        'rank20': cmc_scores[19] * 100,
        'cmc_curve': cmc_scores * 100,
        'time': extraction_time
    }
    
    # Print results
    print(f"\n{'─'*60}")
    print(f"{'Metric':<20} {'Value':<15}")
    print(f"{'─'*60}")
    print(f"{'mAP':<20} {results['mAP']:>14.2f}%")
    print(f"{'Rank-1':<20} {results['rank1']:>14.2f}%")
    print(f"{'Rank-5':<20} {results['rank5']:>14.2f}%")
    print(f"{'Rank-10':<20} {results['rank10']:>14.2f}%")
    print(f"{'Rank-20':<20} {results['rank20']:>14.2f}%")
    print(f"{'─'*60}")
    print(f"{'Inference Time':<20} {results['time']:>11.2f}s")
    print(f"{'─'*60}\n")
    
    return results

# ================================================================================
# PART 5: VISUALIZATION
# ================================================================================

def plot_training_curves(histories_dict):
    """Plot training curves for all models"""
    fig, axes = plt.subplots(1, len(histories_dict), figsize=(6*len(histories_dict), 5))
    
    if len(histories_dict) == 1:
        axes = [axes]
    
    for idx, (model_name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history['train_loss']) + 1)
        
        axes[idx].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
        axes[idx].plot(epochs, history['val_loss'], 'r-s', label='Val Loss', linewidth=2)
        axes[idx].set_xlabel('Epoch', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Loss', fontsize=12, fontweight='bold')
        axes[idx].set_title(f'{model_name} Training', fontsize=14, fontweight='bold')
        axes[idx].legend(fontsize=10)
        axes[idx].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves_all_models.png', dpi=300, bbox_inches='tight')
    plt.show()


def plot_comparison(results_list):
    """Plot comprehensive comparison"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. CMC Curves
    for results in results_list:
        ax1.plot(range(1, 51), results['cmc_curve'], marker='o', 
                markevery=5, linewidth=2.5, label=results['model_name'])
    ax1.set_xlabel('Rank', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Recognition Rate (%)', fontsize=12, fontweight='bold')
    ax1.set_title('CMC Curves Comparison', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(alpha=0.3)
    ax1.set_xlim([1, 50])
    
    # 2. Bar Chart
    models = [r['model_name'] for r in results_list]
    metrics = ['mAP', 'rank1', 'rank5', 'rank10']
    metric_labels = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10']
    
    x = np.arange(len(metric_labels))
    width = 0.8 / len(results_list)
    
    for idx, results in enumerate(results_list):
        values = [results[m] for m in metrics]
        offset = (idx - len(results_list)/2 + 0.5) * width
        bars = ax2.bar(x + offset, values, width, label=results['model_name'], alpha=0.8)
        
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom', fontsize=9)
    
    ax2.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Performance Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metric_labels, fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Inference Time
    times = [r['time'] for r in results_list]
    bars = ax3.bar(models, times, alpha=0.8, color=['steelblue', 'coral', 'mediumseagreen'])
    for bar, time in zip(bars, times):
        ax3.text(bar.get_x() + bar.get_width()/2., time,
                f'{time:.1f}s', ha='center', va='bottom', fontsize=11)
    ax3.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax3.set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Radar Chart
    categories = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10', 'Rank-20']
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    
    ax4 = plt.subplot(224, projection='polar')
    
    for results in results_list:
        values = [results['mAP'], results['rank1'], results['rank5'], 
                 results['rank10'], results['rank20']]
        values += values[:1]
        ax4.plot(angles, values, 'o-', linewidth=2, label=results['model_name'])
        ax4.fill(angles, values, alpha=0.15)
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories, fontsize=10)
    ax4.set_ylim(0, 100)
    ax4.set_title('Multi-Metric Radar', fontsize=14, fontweight='bold', pad=20)
    ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    ax4.grid(True)
    
    plt.tight_layout()
    plt.savefig('comprehensive_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def create_summary_table(results_list):
    """Create summary table"""
    df = pd.DataFrame([{
        'Model': r['model_name'],
        'mAP (%)': f"{r['mAP']:.2f}",
        'Rank-1 (%)': f"{r['rank1']:.2f}",
        'Rank-5 (%)': f"{r['rank5']:.2f}",
        'Rank-10 (%)': f"{r['rank10']:.2f}",
        'Rank-20 (%)': f"{r['rank20']:.2f}",
        'Time (s)': f"{r['time']:.2f}"
    } for r in results_list])
    
    print("\n" + "="*100)
    print("FINAL RESULTS SUMMARY")
    print("="*100)
    print(df.to_string(index=False))
    print("="*100 + "\n")
    
    df.to_csv('final_results_summary.csv', index=False)
    return df

# ================================================================================
# PART 6: MAIN EXECUTION
# ================================================================================

print("\n" + "="*80)
print("PART 6: TRAINING AND EVALUATION PIPELINE")
print("="*80)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

# Storage
trained_models = {}
training_histories = {}
evaluation_results = []

# -------------------- TRAIN OSNet --------------------
print("\n" + "#"*80)
print("# TRAINING OSNET")
print("#"*80)

model_osnet = build_osnet()
model_osnet, history_osnet = train_model(
    model_osnet, train_loader, val_loader, 
    model_name='OSNet',
    num_epochs=5,
    lr=0.0003
)
trained_models['OSNet'] = model_osnet
training_histories['OSNet'] = history_osnet

# Save model
torch.save(model_osnet.state_dict(), 'osnet_market1501.pth')

# -------------------- TRAIN GaitSet --------------------
print("\n" + "#"*80)
print("# TRAINING GAITSET")
print("#"*80)

model_gaitset = build_gaitset()
model_gaitset, history_gaitset = train_model(
    model_gaitset, train_loader, val_loader,
    model_name='GaitSet',
    num_epochs=15,
    lr=0.0003
)
trained_models['GaitSet'] = model_gaitset
training_histories['GaitSet'] = history_gaitset

# Save model
torch.save(model_gaitset.state_dict(), 'gaitset_market1501.pth')

# -------------------- TRAIN FaceNet --------------------
print("\n" + "#"*80)
print("# TRAINING FACENET")
print("#"*80)

model_facenet = build_facenet()
model_facenet, history_facenet = train_model(
    model_facenet, train_loader, val_loader,
    model_name='FaceNet',
    num_epochs=5,
    lr=0.0001  # Lower LR for pretrained model
)
trained_models['FaceNet'] = model_facenet
training_histories['FaceNet'] = history_facenet

# Save model
torch.save(model_facenet.state_dict(), 'facenet_market1501.pth')

# -------------------- PLOT TRAINING CURVES --------------------
print("\nPlotting training curves...")
plot_training_curves(training_histories)

In [ ]:
# ================================================================================
# ADDITION FOR REVIEWER 2: EXHAUSTIVE MODALITY COMBINATIONS ABLATION (MARKET-1501)
# ================================================================================

def evaluate_feature_matrix(q_feats, g_feats, q_pids, g_pids, q_camids, g_camids):
    """Computes exact Re-ID metrics matching your pipeline's evaluation logic"""
    m, n = q_feats.size(0), g_feats.size(0)
    
    # Compute Euclidean distance matrix using optimized addmm_
    dist_mat = torch.pow(q_feats, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    max_rank = 50
    cmc_scores = np.zeros(max_rank)
    ap_scores = []
    
    for i in range(len(q_pids)):
        q_pid, q_cam = q_pids[i], q_camids[i]
        q_dist = dist_mat[i]
        
        sorted_indices = np.argsort(q_dist)
        sorted_pids = g_pids[sorted_indices]
        sorted_cams = g_camids[sorted_indices]
        
        junk_mask = (sorted_pids == q_pid) & (sorted_cams == q_cam)
        keep_mask = ~junk_mask
        matches = (sorted_pids[keep_mask] == q_pid).astype(np.int32)
        
        if len(matches) == 0:
            continue
            
        # CMC Score aggregation
        raw_cmc = matches.cumsum()
        raw_cmc[raw_cmc > 1] = 1
        cmc_padded = np.ones(max_rank) * raw_cmc[-1] if len(raw_cmc) > 0 else np.zeros(max_rank)
        cmc_padded[:min(len(raw_cmc), max_rank)] = raw_cmc[:max_rank]
        cmc_scores += cmc_padded
        
        # Average Precision (AP) calculation
        num_rel = matches.sum()
        if num_rel > 0:
            tmp_cmc = matches.cumsum()
            tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
            ap = np.sum(np.array(tmp_cmc) * matches) / num_rel
            ap_scores.append(ap)
            
    cmc_scores /= len(q_pids)
    return np.mean(ap_scores) * 100, cmc_scores[0] * 100, cmc_scores[4] * 100


def run_market1501_modality_ablation(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset):
    print("\n" + "="*80)
    print("RUNNING EXHAUSTIVE MODALITY PAIR TESTING (REVIEWER 2 REBUTTAL ARTIFACT)")
    print("="*80)
    
    # Explicitly load trained state dictionaries from disk to prevent uninitialized error states
    print("-> Loading trained model parameters from local directory...")
    model_osnet.load_state_dict(torch.load('osnet_market1501.pth', map_location=device))
    model_gaitset.load_state_dict(torch.load('gaitset_market1501.pth', map_location=device))
    model_facenet.load_state_dict(torch.load('facenet_market1501.pth', map_location=device))
    
    model_osnet.eval()
    model_gaitset.eval()
    model_facenet.eval()
    
    # 1. Base Feature Cache Loop
    def cache_dataset_features(dataset, label):
        loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
        app_cache, gait_cache, face_cache = [], [], []
        pids, camids = [], []
        
        with torch.no_grad():
            for imgs, pid, camid in tqdm(loader, desc=f"Extracting base embeddings for {label}"):
                imgs = imgs.to(device)
                
                # Single pass across all three architectures
                out_os = model_osnet(imgs)
                out_gt = model_gaitset(imgs)
                out_fc = model_facenet(imgs)
                
                f_os = out_os[0] if isinstance(out_os, (tuple, list)) else out_os
                f_gt = out_gt[0] if isinstance(out_gt, (tuple, list)) else out_gt
                f_fc = out_fc[0] if isinstance(out_fc, (tuple, list)) else out_fc
                
                # Apply standard L2 normalization on isolated biometric spaces before fusion
                f_os = F.normalize(f_os, p=2, dim=1).cpu()
                f_gt = F.normalize(f_gt, p=2, dim=1).cpu()
                f_fc = F.normalize(f_fc, p=2, dim=1).cpu()
                
                app_cache.append(f_os)
                gait_cache.append(f_gt)
                face_cache.append(f_fc)
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
                
        return torch.cat(app_cache, 0), torch.cat(gait_cache, 0), torch.cat(face_cache, 0), np.array(pids), np.array(camids)

    # Process matrices to system memory
    q_app, q_gait, q_face, q_pids, q_camids = cache_dataset_features(query_dataset, "Query Set")
    g_app, g_gait, g_face, g_pids, g_camids = cache_dataset_features(gallery_dataset, "Gallery Set")
    
    # 2. Define the exact comparison permutations
    combinations = {
        "Appearance + Face (OSNet + FaceNet)": {
            "query": torch.cat([q_app, q_face], dim=1),
            "gallery": torch.cat([g_app, g_face], dim=1)
        },
        "Appearance + Gait (OSNet + GaitSet)": {
            "query": torch.cat([q_app, q_gait], dim=1),
            "gallery": torch.cat([g_app, g_gait], dim=1)
        },
        "Face + Gait (FaceNet + GaitSet)": {
            "query": torch.cat([q_face, q_gait], dim=1),
            "gallery": torch.cat([g_face, g_gait], dim=1)
        },
        "Proposed Pipeline (All 3 Modalities)": {
            "query": torch.cat([q_app, q_face, q_gait], dim=1),
            "gallery": torch.cat([g_app, g_face, g_gait], dim=1)
        }
    }
    
    ablation_metrics = []
    
    # 3. Process matrix combinations sequentially
    print("\n-> Measuring distances across feature spaces...")
    for combo_name, tensors in combinations.items():
        # Force L2 re-normalization on the joint vector space to guarantee accurate cosine distances
        q_fused = F.normalize(tensors["query"], p=2, dim=1)
        g_fused = F.normalize(tensors["gallery"], p=2, dim=1)
        
        mAP, rank1, rank5 = evaluate_feature_matrix(
            q_fused, g_fused, q_pids, g_pids, q_camids, g_camids
        )
        
        ablation_metrics.append({
            "Biometric Mix": combo_name,
            "mAP (%)": mAP,
            "Rank-1 (%)": rank1,
            "Rank-5 (%)": rank5
        })
        print(f"   ✓ Extracted metrics for: {combo_name}")

    # 4. Generate the Manuscript Table Markdown
    df_ablation = pd.DataFrame(ablation_metrics)
    print("\n" + "="*80)
    print("NEW MANUSCRIPT ABLATION TABLE FOR YOUR RESPONSE TO EDITORS")
    print("="*80)
    print(df_ablation.to_string(index=False))
    print("="*80)
    df_ablation.to_csv('modality_combinations_market1501.csv', index=False)

    # 5. Draw Response Chart
    plt.figure(figsize=(12, 6))
    x_indices = np.arange(len(df_ablation))
    bar_width = 0.35
    
    plt.bar(x_indices - bar_width/2, df_ablation["mAP (%)"], bar_width, label='mAP (%)', color='#e74c3c', alpha=0.85)
    plt.bar(x_indices + bar_width/2, df_ablation["Rank-1 (%)"], bar_width, label='Rank-1 (%)', color='#34495e', alpha=0.85)
    
    plt.title('Modality Mix Performance Comparison (Market-1501)', fontsize=14, fontweight='bold')
    plt.xlabel('Assembled Feature Space Configuration', fontsize=12)
    plt.ylabel('Evaluation Score (%)', fontsize=12)
    plt.xticks(x_indices, df_ablation["Biometric Mix"], rotation=12, ha='right')
    plt.ylim(0, 110)
    plt.legend(loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Inject text annotations on the graph bars
    for idx, row in df_ablation.iterrows():
        plt.text(idx - bar_width/2, row["mAP (%)"] + 1, f'{row["mAP (%)"]:.1f}', ha='center', va='bottom', fontsize=10)
        plt.text(idx + bar_width/2, row["Rank-1 (%)"] + 1, f'{row["Rank-1 (%)"]:.1f}', ha='center', va='bottom', fontsize=10)
        
    plt.tight_layout()
    plt.savefig('modality_combinations_market1501.png', dpi=300)
    plt.show()
    print("✓ Rebuttal plot artifact successfully generated as 'modality_combinations_market1501.png'.")

# Execute deployment hook matching your workspace vars
run_market1501_modality_ablation(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset)

In [ ]:
# ================================================================================
# ADDITION FOR REVIEWER 1: MTCNN THRESHOLD SENSITIVITY & DETECTION RATE ANALYSIS
# ================================================================================

from facenet_pytorch import MTCNN
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from PIL import Image

def evaluate_with_gate(q_app, q_gait, q_face, q_probs, g_app, g_gait, g_face, g_probs, 
                        q_pids, g_pids, q_camids, g_camids, tau):
    """Evaluates the pipeline by zeroing out the face stream where confidence < tau"""
    
    # Apply gate mask: if confidence < tau, replace face feature with a zero vector
    q_mask = (q_probs >= tau).astype(np.float32)[:, np.newaxis]
    g_mask = (g_probs >= tau).astype(np.float32)[:, np.newaxis]
    
    q_face_gated = q_face * torch.from_numpy(q_mask)
    g_face_gated = g_face * torch.from_numpy(g_mask)
    
    # Concatenate features to assemble the final multi-modal descriptors
    q_fused = torch.cat([q_app, q_face_gated, q_gait], dim=1)
    g_fused = torch.cat([g_app, g_face_gated, g_gait], dim=1)
    
    # Re-normalize the combined space to ensure correct cosine/Euclidean distance metrics
    q_fused = F.normalize(q_fused, p=2, dim=1)
    g_fused = F.normalize(g_fused, p=2, dim=1)
    
    m, n = q_fused.size(0), g_fused.size(0)
    dist_mat = torch.pow(q_fused, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_fused, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_fused, g_fused.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    max_rank = 50
    cmc_scores = np.zeros(max_rank)
    ap_scores = []
    
    for i in range(len(q_pids)):
        q_pid, q_cam = q_pids[i], q_camids[i]
        q_dist = dist_mat[i]
        
        sorted_indices = np.argsort(q_dist)
        sorted_pids = g_pids[sorted_indices]
        sorted_cams = g_camids[sorted_indices]
        
        junk_mask = (sorted_pids == q_pid) & (sorted_cams == q_cam)
        keep_mask = ~junk_mask
        matches = (sorted_pids[keep_mask] == q_pid).astype(np.int32)
        
        if len(matches) == 0:
            continue
            
        raw_cmc = matches.cumsum()
        raw_cmc[raw_cmc > 1] = 1
        cmc_padded = np.ones(max_rank) * raw_cmc[-1] if len(raw_cmc) > 0 else np.zeros(max_rank)
        cmc_padded[:min(len(raw_cmc), max_rank)] = raw_cmc[:max_rank]
        cmc_scores += cmc_padded
        
        num_rel = matches.sum()
        if num_rel > 0:
            tmp_cmc = matches.cumsum()
            tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
            ap = np.sum(np.array(tmp_cmc) * matches) / num_rel
            ap_scores.append(ap)
            
    cmc_scores /= len(q_pids)
    return np.mean(ap_scores) * 100, cmc_scores[0] * 100


def run_mtcnn_sensitivity_analysis(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset):
    print("\n" + "="*80)
    print("RUNNING MTCNN THRESHOLD SENSITIVITY Sweeps (REVIEWER 1 REBUTTAL)")
    print("="*80)
    
    # Initialize the detector using your active processing hardware
    mtcnn = MTCNN(keep_all=False, select_largest=True, device=device)
    
    model_osnet.eval()
    model_gaitset.eval()
    model_facenet.eval()
    
    # 1. Extraction Helper to collect features and raw MTCNN confidence scores
    def extract_and_get_confidence(dataset, name):
        loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
        app_list, gait_list, face_list, probs_list = [], [], [], []
        pids, camids = [], []
        
        # We loop over indices to read the original raw image files for MTCNN accuracy
        idx = 0
        with torch.no_grad():
            for imgs, pid, camid in tqdm(loader, desc=f"Extracting Embeddings & Face Scores ({name})"):
                imgs = imgs.to(device)
                
                out_os = model_osnet(imgs)
                out_gt = model_gaitset(imgs)
                out_fc = model_facenet(imgs)
                
                f_os = F.normalize(out_os[0] if isinstance(out_os, tuple) else out_os, dim=1).cpu()
                f_gt = F.normalize(out_gt[0] if isinstance(out_gt, tuple) else out_gt, dim=1).cpu()
                f_fc = F.normalize(out_fc[0] if isinstance(out_fc, tuple) else out_fc, dim=1).cpu()
                
                app_list.append(f_os)
                gait_list.append(f_gt)
                face_list.append(f_fc)
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
                
                # Fetch MTCNN confidence score for each image in the current batch
                for _ in range(imgs.size(0)):
                    img_path = dataset.data[idx][0]
                    raw_pil_image = Image.open(img_path).convert('RGB')
                    
                    # Detect face bounding boxes and confidences
                    _, probs = mtcnn.detect(raw_pil_image)
                    
                    # Store the highest face probability or 0.0 if no face is detected
                    max_prob = probs[0] if (probs is not None and len(probs) > 0 and probs[0] is not None) else 0.0
                    probs_list.append(max_prob)
                    idx += 1
                    
        return (torch.cat(app_list, 0), torch.cat(gait_list, 0), torch.cat(face_list, 0), 
                np.array(probs_list), np.array(pids), np.array(camids))

    # Populate feature and score matrices
    q_app, q_gait, q_face, q_probs, q_pids, q_camids = extract_and_get_confidence(query_dataset, "Query")
    g_app, g_gait, g_face, g_probs, g_pids, g_camids = extract_and_get_confidence(gallery_dataset, "Gallery")
    
    # 2. Sweep across different threshold candidates
    threshold_sweeps = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]
    sensitivity_records = []
    
    print("\n-> Sweeping through threshold constraints...")
    for tau in threshold_sweeps:
        # Compute empirical detection rate (fraction of queries passing the threshold)
        empirical_det_rate = (q_probs >= tau).mean() * 100
        
        mAP, rank1 = evaluate_with_gate(
            q_app, q_gait, q_face, q_probs, g_app, g_gait, g_face, g_probs,
            q_pids, g_pids, q_camids, g_camids, tau
        )
        
        sensitivity_records.append({
            "Threshold (tau)": tau,
            "Empirical Detection Rate (%)": empirical_det_rate,
            "System mAP (%)": mAP,
            "System Rank-1 (%)": rank1
        })
        print(f"   ✓ Completed evaluation for threshold tau = {tau:.2f}")
        
    df_sensitivity = pd.DataFrame(sensitivity_records)
    
    # 3. Print the Rebuttal Table
    print("\n" + "="*80)
    print("NEW MANUSCRIPT SENSITIVITY TABLE FOR RESPONSE DOCUMENT")
    print("="*80)
    print(df_sensitivity.to_string(index=False))
    print("="*80)
    df_sensitivity.to_csv('mtcnn_threshold_sensitivity_market1501.csv', index=False)
    
    # 4. Plot Dual-Axis Sensitivity Figure
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    color = '#2c3e50'
    ax1.set_xlabel('MTCNN Confidence Threshold ($\\tau_{face}$)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Empirical Face Detection Rate (%)', color=color, fontsize=12, fontweight='bold')
    line1 = ax1.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["Empirical Detection Rate (%)"], 
                     'o--', color=color, linewidth=2, label='Face Detection Rate (%)')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    # Instantiate a second axis sharing the same x-axis for tracking accuracies
    ax2 = ax1.twinx()  
    color = '#e74c3c'
    ax2.set_ylabel('Pipeline Accuracy (%)', color=color, fontsize=12, fontweight='bold')
    line2 = ax2.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["System mAP (%)"], 
                     's-', color=color, linewidth=2, label='Ensemble mAP (%)')
    line3 = ax2.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["System Rank-1 (%)"], 
                     '^-', color='#3498db', linewidth=2, label='Ensemble Rank-1 (%)')
    ax2.tick_params(axis='y', labelcolor=color)
    
    # Merge legends from both axes
    lines = line1 + line2 + line3
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='lower left')
    
    plt.title('Impact of MTCNN Gate Threshold ($\\tau_{face}$) on Detection Rate and Re-ID Performance', 
              fontsize=13, fontweight='bold', pad=15)
    fig.tight_layout()
    plt.savefig('mtcnn_gate_sensitivity_analysis.png', dpi=300)
    plt.show()
    print("✓ Rebuttal sensitivity visualization saved as 'mtcnn_gate_sensitivity_analysis.png'.")

# Execute deployment sweep
run_mtcnn_sensitivity_analysis(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset)

In [ ]:
df_sensitivity = pd.DataFrame(sensitivity_records)

# 3. Print the Rebuttal Table
print("\n" + "="*80)
print("NEW MANUSCRIPT SENSITIVITY TABLE FOR RESPONSE DOCUMENT")
print("="*80)
print(df_sensitivity.to_string(index=False))
print("="*80)
df_sensitivity.to_csv('mtcnn_threshold_sensitivity_market1501.csv', index=False)

# 4. Plot Dual-Axis Sensitivity Figure
fig, ax1 = plt.subplots(figsize=(10, 6))

color = '#2c3e50'
ax1.set_xlabel('MTCNN Confidence Threshold ($\\tau_{face}$)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Empirical Face Detection Rate (%)', color=color, fontsize=12, fontweight='bold')
line1 = ax1.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["Empirical Detection Rate (%)"], 
                 'o--', color=color, linewidth=2, label='Face Detection Rate (%)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle=':', alpha=0.6)

# Instantiate a second axis sharing the same x-axis for tracking accuracies
ax2 = ax1.twinx()  
color = '#e74c3c'
ax2.set_ylabel('Pipeline Accuracy (%)', color=color, fontsize=12, fontweight='bold')
line2 = ax2.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["System mAP (%)"], 
                 's-', color=color, linewidth=2, label='Ensemble mAP (%)')
line3 = ax2.plot(df_sensitivity["Threshold (tau)"], df_sensitivity["System Rank-1 (%)"], 
                 '^-', color='#3498db', linewidth=2, label='Ensemble Rank-1 (%)')
ax2.tick_params(axis='y', labelcolor=color)

# Merge legends from both axes
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower left')

plt.title('Impact of MTCNN Gate Threshold ($\\tau_{face}$) on Detection Rate and Re-ID Performance', 
          fontsize=13, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig('mtcnn_gate_sensitivity_analysis.png', dpi=300)
plt.show()
print("✓ Rebuttal sensitivity visualization saved as 'mtcnn_gate_sensitivity_analysis.png'.")



In [ ]:
# ================================================================================
# ADDITION FOR REVIEWER 2: CROSS-DATASET GENERALIZATION (MARS -> MARKET-1501)
# ================================================================================

def run_cross_dataset_evaluation(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset, batch_size=64):
    print("\n" + "="*80)
    print("EVALUATING CROSS-DATASET GENERALIZATION: MARS MODELS -> MARKET-1501 DATASET")
    print("="*80)
    
    # 1. Load MARS weights with strict=False to ignore final classifier dimension mismatches
    print("-> Loading pre-trained MARS weights into evaluation backbones...")
    try:
        model_osnet.load_state_dict(torch.load('osnet_market1501.pth', map_location=device), strict=False)
        model_gaitset.load_state_dict(torch.load('gaitset_market1501.pth', map_location=device), strict=False)
        model_facenet.load_state_dict(torch.load('facenet_market1501.pth', map_location=device), strict=False)
        print("  ✓ Backbone layers mapped successfully (Classifier heads skipped safely via strict=False).")
    except FileNotFoundError as e:
        print(f"  [ERROR] Missing weight file: {e.filename}. Ensure 'osnet_mars.pth', 'gaitset_mars.pth', and 'facenet_mars.pth' are in your working directory.")
        return

    # Set all architectures to evaluation mode
    model_osnet.eval()
    model_gaitset.eval()
    model_facenet.eval()
    
    start_time = time.time()
    
    # 2. Unified Feature Extraction Function
    def extract_combined_features(dataset, desc):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        combined_features, pids, camids = [], [], []
        
        with torch.no_grad():
            for img, pid, camid in tqdm(loader, desc=desc):
                img = img.to(device)
                
                # Extract raw backbone outputs
                out_osnet = model_osnet(img)
                out_gaitset = model_gaitset(img)
                out_facenet = model_facenet(img)
                
                # Extract vectors handling explicit tuple formatting
                feat_os = out_osnet[0] if isinstance(out_osnet, (tuple, list)) else out_osnet
                feat_gt = out_gaitset[0] if isinstance(out_gaitset, (tuple, list)) else out_gaitset
                feat_fc = out_facenet[0] if isinstance(out_facenet, (tuple, list)) else out_facenet
                
                # L2 normalize individual modalities prior to blending
                feat_os = F.normalize(feat_os, p=2, dim=1)
                feat_gt = F.normalize(feat_gt, p=2, dim=1)
                feat_fc = F.normalize(feat_fc, p=2, dim=1)
                
                # Form the full identity descriptor via concatenation
                fused_descriptor = torch.cat([feat_os, feat_fc, feat_gt], dim=1)
                
                # Re-normalize the joint space vector
                fused_descriptor = F.normalize(fused_descriptor, p=2, dim=1)
                
                combined_features.append(fused_descriptor.cpu())
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
                
        return torch.cat(combined_features, 0), np.array(pids), np.array(camids)
    
    # Extract features from Market-1501 using the MARS weights
    q_feats, q_pids, q_camids = extract_combined_features(query_dataset, "MARS->Market Query")
    g_feats, g_pids, g_camids = extract_combined_features(gallery_dataset, "MARS->Market Gallery")
    
    extraction_time = time.time() - start_time
    
    # 3. Compute Euclidean Distance Matrix
    print("Computing cross-dataset distance matrix...")
    m, n = q_feats.size(0), g_feats.size(0)
    dist_mat = torch.pow(q_feats, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    # 4. Calculate Generalization Metrics
    print("Calculating rank metrics...")
    max_rank = 50
    cmc_scores = np.zeros(max_rank)
    ap_scores = []
    
    for i in range(len(q_pids)):
        q_pid, q_cam = q_pids[i], q_camids[i]
        q_dist = dist_mat[i]
        
        sorted_indices = np.argsort(q_dist)
        sorted_pids = g_pids[sorted_indices]
        sorted_cams = g_camids[sorted_indices]
        
        junk_mask = (sorted_pids == q_pid) & (sorted_cams == q_cam)
        keep_mask = ~junk_mask
        matches = (sorted_pids[keep_mask] == q_pid).astype(np.int32)
        
        if len(matches) == 0:
            continue
        
        # CMC Accumulation
        raw_cmc = matches.cumsum()
        raw_cmc[raw_cmc > 1] = 1
        cmc_padded = np.ones(max_rank) * raw_cmc[-1] if len(raw_cmc) > 0 else np.zeros(max_rank)
        cmc_padded[:min(len(raw_cmc), max_rank)] = raw_cmc[:max_rank]
        cmc_scores += cmc_padded
        
        # AP Accumulation
        num_rel = matches.sum()
        if num_rel > 0:
            tmp_cmc = matches.cumsum()
            tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
            ap = np.sum(np.array(tmp_cmc) * matches) / num_rel
            ap_scores.append(ap)
            
    cmc_scores /= len(q_pids)
    
    # 5. Report Generalization Capacity Metrics
    cross_results = {
        'mAP': np.mean(ap_scores) * 100,
        'rank1': cmc_scores[0] * 100,
        'rank5': cmc_scores[4] * 100,
        'rank10': cmc_scores[9] * 100
    }
    
    print(f"\n{'─'*60}")
    print(f"{'Cross-Dataset Metric (MARS -> Market)':<38} {'Value':<15}")
    print(f"{'─'*60}")
    print(f"{'Generalization mAP':<38} {cross_results['mAP']:>14.2f}%")
    print(f"{'Generalization Rank-1':<38} {cross_results['rank1']:>14.2f}%")
    print(f"{'Generalization Rank-5':<38} {cross_results['rank5']:>14.2f}%")
    print(f"{'Generalization Rank-10':<38} {cross_results['rank10']:>14.2f}%")
    print(f"{'─'*60}")
    print(f"Total Inference Evaluation Time: {extraction_time:.2f} seconds\n")
    
    # Save to CSV file for reference in the point-by-point rebuttal document
    df_cross = pd.DataFrame([cross_results])
    df_cross.to_csv('cross_dataset_mars_to_market_results.csv', index=False)
    print("✓ Generalization performance saved to 'cross_dataset_mars_to_market_results.csv'")

# Run the cross-dataset generalization routine
run_cross_dataset_evaluation(model_osnet, model_gaitset, model_facenet, query_dataset, gallery_dataset)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# 1. Define Attention Module with Dynamic Input Dimension Support
class EnsembleAttention(nn.Module):
    def __init__(self, app_dim, face_dim, gait_dim):
        super(EnsembleAttention, self).__init__()
        # Dynamically set input size based on true backbone outputs
        self.total_dim = app_dim + face_dim + gait_dim
        self.attention_fc = nn.Linear(self.total_dim, 3)
        print(f"-> Attention layer initialized with input size: {self.total_dim}")

    def forward(self, f_app, f_face, f_gait, return_weights=False):
        concat_feat = torch.cat([f_app, f_face, f_gait], dim=1)
        attn_logits = self.attention_fc(concat_feat)
        attn_weights = F.softmax(attn_logits, dim=1)

        w_app = attn_weights[:, 0:1]
        w_face = attn_weights[:, 1:2]
        w_gait = attn_weights[:, 2:3]

        fused_feat = torch.cat([f_app * w_app, f_gait * w_gait, f_face * w_face], dim=1)

        if return_weights:
            return fused_feat, attn_weights
        return fused_feat

# 2. Main workflow function
def train_attention_and_plot(model_osnet, model_gaitset, model_facenet, train_loader, query_dataset):
    print("-> Loading frozen weights from your saved checkpoints...")
    model_osnet.load_state_dict(torch.load('osnet_market1501.pth', map_location=device))
    model_gaitset.load_state_dict(torch.load('gaitset_market1501.pth', map_location=device))
    model_facenet.load_state_dict(torch.load('facenet_market1501.pth', map_location=device))
    
    # Freeze the backbones entirely to save memory and time
    model_osnet.eval()
    model_gaitset.eval()
    model_facenet.eval()
    
    # --- DYNAMIC SHAPE DETECTION ---
    print("-> Detecting backbone feature dimensions dynamically...")
    with torch.no_grad():
        for sample_imgs, _, _, _ in train_loader:
            sample_imgs = sample_imgs.to(device)
            out_os = model_osnet(sample_imgs)
            out_gt = model_gaitset(sample_imgs)
            out_fc = model_facenet(sample_imgs)
            
            f_os = out_os[0] if isinstance(out_os, tuple) else out_os
            f_gt = out_gt[0] if isinstance(out_gt, tuple) else out_gt
            f_fc = out_fc[0] if isinstance(out_fc, tuple) else out_fc
            
            detected_app_dim = f_os.size(1)
            detected_gait_dim = f_gt.size(1)
            detected_face_dim = f_fc.size(1)
            
            print(f"   Detected -> OSNet: {detected_app_dim} | FaceNet: {detected_face_dim} | GaitSet: {detected_gait_dim}")
            break
            
    # Initialize attention fusion block with the exact detected sizes
    attn_module = EnsembleAttention(
        app_dim=detected_app_dim, 
        face_dim=detected_face_dim, 
        gait_dim=detected_gait_dim
    ).to(device)
    
    # Setup a quick alignment optimizer for the 3 weights
    triplet_loss = TripletLoss(margin=0.3)
    optimizer = torch.optim.Adam(attn_module.parameters(), lr=0.001)
    
    print("\n" + "="*60)
    print("STEP 1: TRAINING ATTENTION MODULE (1 QUICK ALIGNMENT EPOCH)")
    print("="*60)
    
    attn_module.train()
    for anchor, positive, negative, _ in tqdm(train_loader, desc="Learning Weights"):
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        
        with torch.no_grad(): # Keep backbones frozen
            a_os, p_os, n_os = model_osnet(anchor), model_osnet(positive), model_osnet(negative)
            a_gt, p_gt, n_gt = model_gaitset(anchor), model_gaitset(positive), model_gaitset(negative)
            a_fc, p_fc, n_fc = model_facenet(anchor), model_facenet(positive), model_facenet(negative)
            
            # Normalization sync handling formatting styles
            a_os = F.normalize(a_os[0] if isinstance(a_os, tuple) else a_os, dim=1)
            p_os = F.normalize(p_os[0] if isinstance(p_os, tuple) else p_os, dim=1)
            n_os = F.normalize(n_os[0] if isinstance(n_os, tuple) else n_os, dim=1)
            
            a_gt = F.normalize(a_gt[0] if isinstance(a_gt, tuple) else a_gt, dim=1)
            p_gt = F.normalize(p_gt[0] if isinstance(p_gt, tuple) else p_gt, dim=1)
            n_gt = F.normalize(n_gt[0] if isinstance(n_gt, tuple) else n_gt, dim=1)
            
            a_fc = F.normalize(a_fc[0] if isinstance(a_fc, tuple) else a_fc, dim=1)
            p_fc = F.normalize(p_fc[0] if isinstance(p_fc, tuple) else p_fc, dim=1)
            n_fc = F.normalize(n_fc[0] if isinstance(n_fc, tuple) else n_fc, dim=1)
        
        optimizer.zero_grad()
        
        # Optimize fusion selection targets
        fused_a = attn_module(a_os, a_fc, a_gt)
        fused_p = attn_module(p_os, p_fc, p_gt)
        fused_n = attn_module(n_os, n_fc, n_gt)
        
        loss = triplet_loss(fused_a, fused_p, fused_n)
        loss.backward()
        optimizer.step()

    print("\n" + "="*60)
    print("STEP 2: EXTRACTING TRAINED ATTENTION WEIGHT DISTRIBUTION")
    print("="*60)
    
    attn_module.eval()
    eval_loader = DataLoader(query_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    all_weights, all_pids = [], []
    
    with torch.no_grad():
        for imgs, pids, _ in tqdm(eval_loader, desc="Evaluating Query Classes"):
            imgs = imgs.to(device)
            
            out_os = model_osnet(imgs)
            out_gt = model_gaitset(imgs)
            out_fc = model_facenet(imgs)
            
            f_os = F.normalize(out_os[0] if isinstance(out_os, tuple) else out_os, dim=1)
            f_gt = F.normalize(out_gt[0] if isinstance(out_gt, tuple) else out_gt, dim=1)
            f_fc = F.normalize(out_fc[0] if isinstance(out_fc, tuple) else out_fc, dim=1)
            
            _, weights = attn_module(f_os, f_fc, f_gt, return_weights=True)
            all_weights.append(weights.cpu().numpy())
            all_pids.extend(pids.tolist())
            
    all_weights = np.vstack(all_weights)
    df_weights = pd.DataFrame({
        'Query_ID': all_pids,
        'Appearance (OSNet)': all_weights[:, 1],
        'Face (FaceNet)': all_weights[:, 0],
        'Gait (GaitSet)': all_weights[:, 2]
    })
    
    # Calculate group means per identity class
    avg_weights = df_weights.groupby('Query_ID').mean()
    
    # Step 3: Draw the final scannable chart for the reviewer
    sample_classes = avg_weights.sample(n=15, random_state=42)
    ax = sample_classes.plot(
        kind='bar', stacked=True, figsize=(14, 6),
        color=['#e74c3c', '#2ecc71', '#f39c12'], alpha=0.85
    )
    
    plt.title('Learned Attention Weights per Query Identity Class (Market-1501)', fontsize=13, fontweight='bold')
    plt.xlabel('Query Identity Class (Person ID)', fontsize=11)
    plt.ylabel('Attention Weight Distribution Score', fontsize=11)
    plt.legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=3, fontsize=10)
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig('learned_attention_ablation.png', dpi=300)
    plt.show()
    
    # Output numerical sample verification text block
    print("\n=== RAW SAMPLE METRICS FOR THE REBUTTAL TEXT ===")
    print(sample_classes.head(5))

# Execute workspace pipeline function
train_attention_and_plot(model_osnet, model_gaitset, model_facenet, train_loader, query_dataset)

In [ ]:
sample_classes = avg_weights.sample(n=15, random_state=42)
ax = sample_classes.plot(
    kind='bar', stacked=True, figsize=(14, 6),
    color=['#e74c3c', '#2ecc71', '#f39c12'], alpha=0.85
)

plt.title('Learned Attention Weights per Query Identity Class (Market-1501)', fontsize=13, fontweight='bold')
plt.xlabel('Query Identity Class (Person ID)', fontsize=11)
plt.ylabel('Attention Weight Distribution Score', fontsize=11)
plt.legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=3, fontsize=10)
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('learned_attention_ablation.png', dpi=300)
plt.show()

# Output numerical sample verification text block
print("\n=== RAW SAMPLE METRICS FOR THE REBUTTAL TEXT ===")
print(sample_classes.head(5))

In [ ]:
print("\nPlotting training curves...")
plot_training_curves(training_histories)

In [ ]:
def plot_comparison(results_list):
    """Plot comprehensive comparison"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. CMC Curves
    for results in results_list:
        ax1.plot(range(1, 21), results['cmc_curve'], marker='o', 
                markevery=5, linewidth=2.5, label=results['model_name'])
    ax1.set_xlabel('Rank', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Recognition Rate (%)', fontsize=12, fontweight='bold')
    ax1.set_title('CMC Curves Comparison', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(alpha=0.3)
    ax1.set_xlim([1, 20])
    
    # 2. Bar Chart
    models = [r['model_name'] for r in results_list]
    metrics = ['mAP', 'rank1', 'rank5', 'rank10']
    metric_labels = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10']
    
    x = np.arange(len(metric_labels))
    width = 0.8 / len(results_list)
    
    for idx, results in enumerate(results_list):
        values = [results[m] for m in metrics]
        offset = (idx - len(results_list)/2 + 0.5) * width
        bars = ax2.bar(x + offset, values, width, label=results['model_name'], alpha=0.8)
        
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom', fontsize=9)
    
    ax2.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Performance Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metric_labels, fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Inference Time
    times = [r['time'] for r in results_list]
    bars = ax3.bar(models, times, alpha=0.8, color=['steelblue', 'coral', 'mediumseagreen'])
    for bar, time in zip(bars, times):
        ax3.text(bar.get_x() + bar.get_width()/2., time,
                f'{time:.1f}s', ha='center', va='bottom', fontsize=11)
    ax3.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax3.set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Radar Chart
    categories = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10', 'Rank-20']
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    
    ax4 = plt.subplot(224, projection='polar')
    
    for results in results_list:
        values = [results['mAP'], results['rank1'], results['rank5'], 
                 results['rank10'], results['rank20']]
        values += values[:1]
        ax4.plot(angles, values, 'o-', linewidth=2, label=results['model_name'])
        ax4.fill(angles, values, alpha=0.15)
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories, fontsize=10)
    ax4.set_ylim(0, 100)
    ax4.set_title('Multi-Metric Radar', fontsize=14, fontweight='bold', pad=20)
    ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    ax4.grid(True)
    
    plt.tight_layout()
    plt.savefig('comprehensive_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def create_summary_table(results_list):
    """Create summary table"""
    df = pd.DataFrame([{
        'Model': r['model_name'],
        'mAP (%)': f"{r['mAP']:.2f}",
        'Rank-1 (%)': f"{r['rank1']:.2f}",
        'Rank-5 (%)': f"{r['rank5']:.2f}",
        'Rank-10 (%)': f"{r['rank10']:.2f}",
        'Rank-20 (%)': f"{r['rank20']:.2f}",
        'Time (s)': f"{r['time']:.2f}"
    } for r in results_list])
    
    print("\n" + "="*100)
    print("FINAL RESULTS SUMMARY")
    print("="*100)
    print(df.to_string(index=False))
    print("="*100 + "\n")
    
    df.to_csv('final_results_summary.csv', index=False)
    return df


In [ ]:
for model_name, model in trained_models.items():
    print(f"\nEvaluating {model_name}...")
    results = evaluate_model(model, query_dataset, gallery_dataset, model_name)
    evaluation_results.append(results)
    

In [ ]:
plot_comparison(results_list)
create_summary_table(results_list)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a new standalone figure and axes
fig, ax = plt.subplots(figsize=(10, 6))

# Filter data (Assuming 'evaluation_results' is already defined)
ensemble_only = [r for r in evaluation_results if 'Ensemble' in r['model_name']]
x = np.arange(len(ensemble_only))
width = 0.15
metrics = ['mAP', 'rank1', 'rank5', 'rank10', 'rank20']
metric_labels = ['mAP', 'R-1', 'R-5', 'R-10', 'R-20']
colors_metric = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

# Plotting loop
for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    values = [r[metric] for r in ensemble_only]
    ax.bar(x + i*width, values, width, label=label, 
           color=colors_metric[i], alpha=0.8)

# Formatting
ax.set_xlabel('Ensemble Technique', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Ensemble Techniques Detailed Comparison', 
             fontsize=14, fontweight='bold')

# Set y-axis to start from 60
ax.set_ylim(bottom=60) 

# Ticks
ax.set_xticks(x + width * 2)
ax.set_xticklabels([r['model_name'].replace('Ensemble-', '') 
                    for r in ensemble_only], rotation=45, ha='right')

# Legend and Grid
ax.legend(fontsize=10, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.2))
ax.grid(axis='y', alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()

In [ ]:
"""
================================================================================
ENSEMBLE MODEL FOR PERSON RE-IDENTIFICATION
Multiple Fusion Techniques Comparison
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import seaborn as sns

# ================================================================================
# PART 7: ENSEMBLE MODELS
# ================================================================================

print("\n" + "="*80)
print("PART 7: ENSEMBLE FUSION TECHNIQUES")
print("="*80)

class EnsembleReID(nn.Module):
    """Ensemble model with multiple fusion strategies"""
    
    def __init__(self, model_osnet, model_gaitset, model_facenet, 
                 fusion_type='weighted_avg', weights=None):
        super(EnsembleReID, self).__init__()
        
        self.model_osnet = model_osnet
        self.model_gaitset = model_gaitset
        self.model_facenet = model_facenet
        self.fusion_type = fusion_type
        
        # Set all models to eval mode
        self.model_osnet.eval()
        self.model_gaitset.eval()
        self.model_facenet.eval()
        
        # Freeze base models
        for param in self.model_osnet.parameters():
            param.requires_grad = False
        for param in self.model_gaitset.parameters():
            param.requires_grad = False
        for param in self.model_facenet.parameters():
            param.requires_grad = False
        
        # Get feature dimensions (typical for these models)
        # OSNet: 512, GaitSet: 256, FaceNet: 512
        self.osnet_dim = 512
        self.gaitset_dim = 256
        self.facenet_dim = 512
        
        # Initialize fusion-specific components
        if fusion_type == 'weighted_avg':
            # Learnable weights for weighted average
            if weights is None:
                self.weights = nn.Parameter(torch.ones(3) / 3)
            else:
                self.weights = nn.Parameter(torch.tensor(weights, dtype=torch.float32))
        
        elif fusion_type == 'attention':
            # Attention-based fusion
            total_dim = self.osnet_dim + self.gaitset_dim + self.facenet_dim
            self.attention_fc = nn.Sequential(
                nn.Linear(total_dim, 512),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(512, 3),
                nn.Softmax(dim=1)
            )
        
        elif fusion_type == 'mlp':
            # MLP fusion
            total_dim = self.osnet_dim + self.gaitset_dim + self.facenet_dim
            self.fusion_mlp = nn.Sequential(
                nn.Linear(total_dim, 1024),
                nn.BatchNorm1d(1024),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(1024, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(512, 512)
            )
        
        elif fusion_type == 'gated':
            # Gated fusion
            self.gate_osnet = nn.Sequential(
                nn.Linear(self.osnet_dim, self.osnet_dim),
                nn.Sigmoid()
            )
            self.gate_gaitset = nn.Sequential(
                nn.Linear(self.gaitset_dim, self.gaitset_dim),
                nn.Sigmoid()
            )
            self.gate_facenet = nn.Sequential(
                nn.Linear(self.facenet_dim, self.facenet_dim),
                nn.Sigmoid()
            )
            # Project to common dimension
            self.proj_osnet = nn.Linear(self.osnet_dim, 512)
            self.proj_gaitset = nn.Linear(self.gaitset_dim, 512)
            self.proj_facenet = nn.Linear(self.facenet_dim, 512)
    
    def extract_features(self, x):
        """Extract features from all three models"""
        with torch.no_grad():
            # OSNet
            osnet_out = self.model_osnet(x)
            osnet_feat = osnet_out[0] if isinstance(osnet_out, tuple) else osnet_out
            
            # GaitSet
            gaitset_out = self.model_gaitset(x)
            gaitset_feat = gaitset_out[0] if isinstance(gaitset_out, tuple) else gaitset_out
            
            # FaceNet
            facenet_out = self.model_facenet(x)
            facenet_feat = facenet_out[0] if isinstance(facenet_out, tuple) else facenet_out
        
        return osnet_feat, gaitset_feat, facenet_feat
    
    def forward(self, x):
        # Extract features
        osnet_feat, gaitset_feat, facenet_feat = self.extract_features(x)
        
        # Normalize features
        osnet_feat = F.normalize(osnet_feat, p=2, dim=1)
        gaitset_feat = F.normalize(gaitset_feat, p=2, dim=1)
        facenet_feat = F.normalize(facenet_feat, p=2, dim=1)
        
        # Apply fusion strategy
        if self.fusion_type == 'simple_avg':
            # Simple average (equal weights)
            # Project to common dimension first
            osnet_proj = osnet_feat
            gaitset_proj = F.adaptive_avg_pool1d(gaitset_feat.unsqueeze(1), 
                                                  self.osnet_dim).squeeze(1)
            facenet_proj = facenet_feat
            fused = (osnet_proj + gaitset_proj + facenet_proj) / 3
        
        elif self.fusion_type == 'weighted_avg':
            # Weighted average with learnable weights
            weights = F.softmax(self.weights, dim=0)
            osnet_proj = osnet_feat * weights[0]
            gaitset_proj = F.adaptive_avg_pool1d(gaitset_feat.unsqueeze(1), 
                                                  self.osnet_dim).squeeze(1) * weights[1]
            facenet_proj = facenet_feat * weights[2]
            fused = osnet_proj + gaitset_proj + facenet_proj
        
        elif self.fusion_type == 'concatenation':
            # Simple concatenation
            fused = torch.cat([osnet_feat, gaitset_feat, facenet_feat], dim=1)
        
        elif self.fusion_type == 'attention':
            # Attention-based fusion
            concat_feat = torch.cat([osnet_feat, gaitset_feat, facenet_feat], dim=1)
            attention_weights = self.attention_fc(concat_feat)
            
            osnet_proj = osnet_feat * attention_weights[:, 0:1]
            gaitset_proj = F.adaptive_avg_pool1d(gaitset_feat.unsqueeze(1), 
                                                  self.osnet_dim).squeeze(1) * attention_weights[:, 1:2]
            facenet_proj = facenet_feat * attention_weights[:, 2:3]
            fused = osnet_proj + gaitset_proj + facenet_proj
        
        elif self.fusion_type == 'mlp':
            # MLP fusion
            concat_feat = torch.cat([osnet_feat, gaitset_feat, facenet_feat], dim=1)
            fused = self.fusion_mlp(concat_feat)
        
        elif self.fusion_type == 'gated':
            # Gated fusion
            gate_o = self.gate_osnet(osnet_feat)
            gate_g = self.gate_gaitset(gaitset_feat)
            gate_f = self.gate_facenet(facenet_feat)
            
            osnet_gated = self.proj_osnet(osnet_feat * gate_o)
            gaitset_gated = self.proj_gaitset(gaitset_feat * gate_g)
            facenet_gated = self.proj_facenet(facenet_feat * gate_f)
            
            fused = osnet_gated + gaitset_gated + facenet_gated
        
        else:
            raise ValueError(f"Unknown fusion type: {self.fusion_type}")
        
        # Final normalization
        fused = F.normalize(fused, p=2, dim=1)
        return fused


def evaluate_ensemble(model, query_dataset, gallery_dataset, 
                     model_name, batch_size=64):
    """Evaluate ensemble model"""
    
    print(f"\n{'='*60}")
    print(f"EVALUATING {model_name.upper()}")
    print(f"{'='*60}")
    
    model.eval()
    
    def extract_features(dataset, desc):
        from torch.utils.data import DataLoader
        from tqdm import tqdm
        
        loader = DataLoader(dataset, batch_size=batch_size, 
                          shuffle=False, num_workers=2)
        features, pids, camids = [], [], []
        
        with torch.no_grad():
            for img, pid, camid in tqdm(loader, desc=desc):
                img = img.to(device)
                feat = model(img)
                
                features.append(feat.cpu())
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
        
        return torch.cat(features, 0), np.array(pids), np.array(camids)
    
    # Extract features
    q_feats, q_pids, q_camids = extract_features(query_dataset, f"{model_name} Query")
    g_feats, g_pids, g_camids = extract_features(gallery_dataset, f"{model_name} Gallery")
    
    # Compute distances
    print("Computing distance matrix...")
    m, n = q_feats.size(0), g_feats.size(0)
    dist_mat = torch.pow(q_feats, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    # Calculate metrics
    print("Calculating metrics...")
    max_rank = 50
    cmc_scores = np.zeros(max_rank)
    ap_scores = []
    
    for i in range(len(q_pids)):
        q_pid, q_cam = q_pids[i], q_camids[i]
        q_dist = dist_mat[i]
        
        sorted_indices = np.argsort(q_dist)
        sorted_pids = g_pids[sorted_indices]
        sorted_cams = g_camids[sorted_indices]
        
        junk_mask = (sorted_pids == q_pid) & (sorted_cams == q_cam)
        keep_mask = ~junk_mask
        matches = (sorted_pids[keep_mask] == q_pid).astype(np.int32)
        
        if len(matches) == 0:
            continue
        
        # CMC
        raw_cmc = matches.cumsum()
        raw_cmc[raw_cmc > 1] = 1
        cmc_padded = np.ones(max_rank) * raw_cmc[-1] if len(raw_cmc) > 0 else np.zeros(max_rank)
        cmc_padded[:min(len(raw_cmc), max_rank)] = raw_cmc[:max_rank]
        cmc_scores += cmc_padded
        
        # AP
        num_rel = matches.sum()
        if num_rel > 0:
            tmp_cmc = matches.cumsum()
            tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
            ap = np.sum(np.array(tmp_cmc) * matches) / num_rel
            ap_scores.append(ap)
    
    cmc_scores /= len(q_pids)
    
    results = {
        'model_name': model_name,
        'mAP': np.mean(ap_scores) * 100,
        'rank1': cmc_scores[0] * 100,
        'rank5': cmc_scores[4] * 100,
        'rank10': cmc_scores[9] * 100,
        'rank20': cmc_scores[19] * 100,
        'cmc_curve': cmc_scores * 100
    }
    
    # Print results
    # print(f"\n{'─'*60}")
    # print(f"{'Metric':<20} {'Value':<15}")
    # print(f"{'─'*60}")
    # print(f"{'mAP':<20} {results['mAP']:>14.2f}%")
    # print(f"{'Rank-1':<20} {results['rank1']:>14.2f}%")
    # print(f"{'Rank-5':<20} {results['rank5']:>14.2f}%")
    # print(f"{'Rank-10':<20} {results['rank10']:>14.2f}%")
    # print(f"{'Rank-20':<20} {results['rank20']:>14.2f}%")
    print(f"{'─'*60}\n")
    
    return results


# ================================================================================
# MAIN EXECUTION: COMPARE ENSEMBLE TECHNIQUES
# ================================================================================

print("\n" + "="*80)
print("COMPARING ENSEMBLE TECHNIQUES")
print("="*80)

# Define fusion techniques to compare
fusion_techniques = [
    'simple_avg',
    'weighted_avg',
    'concatenation',
    'attention',
    'mlp',
    'gated'
]

ensemble_results = []

for fusion_type in fusion_techniques:
    print(f"\n{'#'*80}")
    print(f"# TESTING: {fusion_type.upper()} FUSION")
    print(f"{'#'*80}")
    
    # Create ensemble model
    ensemble_model = EnsembleReID(
        trained_models['OSNet'],
        trained_models['GaitSet'],
        trained_models['FaceNet'],
        fusion_type=fusion_type
    ).to(device)
    
    # Evaluate
    results = evaluate_ensemble(
        ensemble_model,
        query_dataset,
        gallery_dataset,
        model_name=f'Ensemble-{fusion_type}',
        batch_size=32
    )
    
    ensemble_results.append(results)
    
    # Save model
    torch.save(ensemble_model.state_dict(), 
              f'ensemble_{fusion_type}_market1501.pth')



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def plot_comparison(results_list):
    """Plot comprehensive comparison"""
    # Create a figure to hold a 2x2 grid of plots
    fig = plt.figure(figsize=(16, 12))
    
    # ---------------------------------------------------------
    # 1. CMC Curves (Top Left)
    # ---------------------------------------------------------
    ax1 = fig.add_subplot(2, 2, 1)
    
    for results in results_list:
        ax1.plot(range(1, 21), results['cmc_curve'], marker='o', 
                 markevery=5, linewidth=2.5, label=results['model_name'])
        
    ax1.set_xlabel('Rank', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Recognition Rate (%)', fontsize=12, fontweight='bold')
    ax1.set_title('CMC Curves Comparison', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    ax1.set_xlim([1, 20]) # Fixed to match the 20 ranks
    
    # ---------------------------------------------------------
    # 2. Bar Chart (Top Right)
    # ---------------------------------------------------------
    ax2 = fig.add_subplot(2, 2, 2)
    
    metrics = ['mAP', 'rank1', 'rank5', 'rank10']
    metric_labels = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10']
    
    x = np.arange(len(metric_labels))
    width = 0.8 / len(results_list)
    
    for idx, results in enumerate(results_list):
        values = [results[m] for m in metrics]
        offset = (idx - len(results_list)/2 + 0.5) * width
        bars = ax2.bar(x + offset, values, width, label=results['model_name'], alpha=0.8)
        
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                     f'{height:.1f}', ha='center', va='bottom', fontsize=9, rotation=90)
    
    ax2.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Performance Comparison', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metric_labels, fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim([0, 105]) # Give headroom for text labels
    
    # ---------------------------------------------------------
    # 3. Inference Time (Bottom Left - Currently Empty/Placeholder)
    # ---------------------------------------------------------
    # ax3 = fig.add_subplot(2, 2, 3)
    # (Uncomment and adjust this section if you add 'time' back to your data)
    
    # ---------------------------------------------------------
    # 4. Radar Chart (Bottom Right)
    # ---------------------------------------------------------
    categories = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10', 'Rank-20']
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    
    # Use polar projection for the radar chart
    ax4 = fig.add_subplot(2, 2, 4, projection='polar')
    
    for results in results_list:
        values = [results['mAP'], results['rank1'], results['rank5'], 
                  results['rank10'], results['rank20']]
        values += values[:1] # Close the polygon
        ax4.plot(angles, values, 'o-', linewidth=2, label=results['model_name'])
        ax4.fill(angles, values, alpha=0.15)
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories, fontsize=10)
    ax4.set_ylim(0, 100)
    ax4.set_title('Multi-Metric Radar', fontsize=14, fontweight='bold', pad=20)
    
    # Adjust legend position so it doesn't overlap the radar
    ax4.legend(loc='center left', bbox_to_anchor=(1.15, 0.5), fontsize=10) 
    
    plt.tight_layout()
    plt.savefig('comprehensive_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# Assuming `ensemble_results` is defined in your environment
plot_comparison(ensemble_results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ================================================================================
# VISUALIZATION: ENSEMBLE COMPARISON
# ================================================================================

print("\n" + "="*80)
print("VISUALIZING ENSEMBLE COMPARISON")
print("="*80)

# Assuming `evaluation_results` (individual models) and `ensemble_results` are already defined.
# If evaluation_results doesn't exist yet in your scope, make sure to initialize it!
evaluation_results.extend(ensemble_results) 

# Initialize the figure with a 2x3 layout
plt.figure(figsize=(22, 12))

# --------------------------------------------------------------------------------
# 1. CMC Curves (Top Left)
# --------------------------------------------------------------------------------
ax1 = plt.subplot(2, 3, 1)
for r in evaluation_results:
    if 'cmc_curve' in r:
        ax1.plot(range(1, 21), r['cmc_curve'], marker='o', 
                 markevery=5, linewidth=2, label=r['model_name'])

ax1.set_xlabel('Rank', fontsize=12, fontweight='bold')
ax1.set_ylabel('Recognition Rate (%)', fontsize=12, fontweight='bold')
ax1.set_title('CMC Curves Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=8, loc='lower right') # Kept small since there are many models
ax1.grid(alpha=0.3)
ax1.set_xlim([1, 20])
ax1.set_ylim([30, 100]) # Adjusted to fit standard ReID CMC curve ranges

# --------------------------------------------------------------------------------
# 2. mAP Comparison (Top Middle)
# --------------------------------------------------------------------------------
ax2 = plt.subplot(2, 3, 2)
models = [r['model_name'] for r in evaluation_results]
maps = [r['mAP'] for r in evaluation_results]
colors_bar = ['steelblue', 'coral', 'mediumseagreen', 'purple', 'orange', 'pink', 'brown', 'gray', 'olive']
bars = ax2.barh(models, maps, color=colors_bar[:len(models)], alpha=0.8)

for bar, val in zip(bars, maps):
    ax2.text(val + 1, bar.get_y() + bar.get_height()/2, 
             f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')
             
ax2.set_xlabel('mAP (%)', fontsize=12, fontweight='bold')
ax2.set_title('Mean Average Precision Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.set_xlim([0, max(maps) + 15]) 

# --------------------------------------------------------------------------------
# 3. Rank-N Metrics Heatmap (Top Right)
# --------------------------------------------------------------------------------
ax3 = plt.subplot(2, 3, 3)
rank_data = []
for r in evaluation_results:
    rank_data.append([r['mAP'], r['rank1'], r['rank5'], r['rank10'], r['rank20']])
    
rank_df = pd.DataFrame(rank_data, 
                       columns=['mAP', 'R-1', 'R-5', 'R-10', 'R-20'],
                       index=[r['model_name'] for r in evaluation_results])
                       
sns.heatmap(rank_df, annot=True, fmt='.2f', cmap='RdYlGn', 
            cbar_kws={'label': 'Score (%)'}, ax=ax3, linewidths=0.5)
ax3.set_title('Performance Heatmap', fontsize=14, fontweight='bold')
ax3.set_xlabel('')
ax3.set_ylabel('')

# --------------------------------------------------------------------------------
# 4. Ensemble Techniques Only - Detailed Comparison (Bottom Left)
# --------------------------------------------------------------------------------
ax4 = plt.subplot(2, 3, 4)
ensemble_only = [r for r in evaluation_results if 'Ensemble' in r['model_name']]
x = np.arange(len(ensemble_only))
width = 0.15
metrics = ['mAP', 'rank1', 'rank5', 'rank10', 'rank20']
metric_labels = ['mAP', 'R-1', 'R-5', 'R-10', 'R-20']
colors_metric = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    values = [r[metric] for r in ensemble_only]
    ax4.bar(x + i*width, values, width, label=label, color=colors_metric[i], alpha=0.8)

ax4.set_xlabel('Ensemble Technique', fontsize=12, fontweight='bold')
ax4.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax4.set_title('Ensemble Techniques Detailed Comparison', fontsize=14, fontweight='bold')
ax4.set_xticks(x + width * 2)
ax4.set_xticklabels([r['model_name'].replace('Ensemble ', '').replace('Ensemble', '').strip() 
                     for r in ensemble_only], rotation=45, ha='right')
ax4.legend(fontsize=10, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.2))
ax4.grid(axis='y', alpha=0.3)

# --------------------------------------------------------------------------------
# 5. Improvement over Best Individual Model (Bottom Middle)
# --------------------------------------------------------------------------------
ax5 = plt.subplot(2, 3, 5)
best_individual = max([r for r in evaluation_results if 'Ensemble' not in r['model_name']], 
                      key=lambda x: x['mAP'])
                      
improvements = []
for r in ensemble_only:
    imp = {
        'technique': r['model_name'].replace('Ensemble ', '').replace('Ensemble', '').strip(),
        'mAP_imp': r['mAP'] - best_individual['mAP'],
        'rank1_imp': r['rank1'] - best_individual['rank1']
    }
    improvements.append(imp)

techniques = [i['technique'] for i in improvements]
map_imps = [i['mAP_imp'] for i in improvements]
rank1_imps = [i['rank1_imp'] for i in improvements]

x = np.arange(len(techniques))
width = 0.35
bars1 = ax5.bar(x - width/2, map_imps, width, label='mAP Improvement', color='steelblue', alpha=0.8)
bars2 = ax5.bar(x + width/2, rank1_imps, width, label='Rank-1 Improvement', color='coral', alpha=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:+.2f}', ha='center', 
                 va='bottom' if height >= 0 else 'top', fontsize=9)

ax5.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax5.set_xlabel('Ensemble Technique', fontsize=12, fontweight='bold')
ax5.set_ylabel('Improvement (%)', fontsize=12, fontweight='bold')
ax5.set_title(f'Improvement over Best Individual ({best_individual["model_name"]})', fontsize=14, fontweight='bold')
ax5.set_xticks(x)
ax5.set_xticklabels(techniques, rotation=45, ha='right')
ax5.legend(fontsize=10)
ax5.grid(axis='y', alpha=0.3)

# --------------------------------------------------------------------------------
# 6. Best Ensemble Highlight (Bottom Right)
# --------------------------------------------------------------------------------
ax6 = plt.subplot(2, 3, 6)
best_ensemble = max(ensemble_only, key=lambda x: x['mAP'])
categories = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10', 'Rank-20']
best_values = [best_ensemble['mAP'], best_ensemble['rank1'], 
               best_ensemble['rank5'], best_ensemble['rank10'], best_ensemble['rank20']]
individual_best_values = [best_individual['mAP'], best_individual['rank1'],
                          best_individual['rank5'], best_individual['rank10'], 
                          best_individual['rank20']]

x = np.arange(len(categories))
width = 0.35
bars1 = ax6.bar(x - width/2, individual_best_values, width, 
                label=best_individual['model_name'], color='lightcoral', alpha=0.8)
bars2 = ax6.bar(x + width/2, best_values, width,
                label=best_ensemble['model_name'], color='mediumseagreen', alpha=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height + 1,
                 f'{height:.1f}', ha='center', va='bottom', fontsize=9)

ax6.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax6.set_title('Best Ensemble vs Best Individual', fontsize=14, fontweight='bold')
ax6.set_xticks(x)
ax6.set_xticklabels(categories)
ax6.legend(fontsize=10)
ax6.grid(axis='y', alpha=0.3)
ax6.set_ylim([0, 110]) 

plt.tight_layout()
plt.savefig('ensemble_comprehensive_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# ================================================================================
# FINAL SUMMARY
# ================================================================================

print("\n" + "="*100)
print("FINAL ENSEMBLE COMPARISON SUMMARY")
print("="*100)

summary_data = []
for r in evaluation_results:
    summary_data.append({
        'Model': r['model_name'],
        'Type': 'Individual' if 'Ensemble' not in r['model_name'] else 'Ensemble',
        'raw_mAP': r['mAP'], 
        'Rank-1 (%)': f"{r['rank1']:.2f}",
        'Rank-5 (%)': f"{r['rank5']:.2f}",
        'Rank-10 (%)': f"{r['rank10']:.2f}",
        'Rank-20 (%)': f"{r['rank20']:.2f}"
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('raw_mAP', ascending=False)
df_summary['mAP (%)'] = df_summary['raw_mAP'].apply(lambda x: f"{x:.2f}")
df_summary = df_summary.drop(columns=['raw_mAP'])

cols = ['Model', 'Type', 'mAP (%)', 'Rank-1 (%)', 'Rank-5 (%)', 'Rank-10 (%)', 'Rank-20 (%)']
df_summary = df_summary[cols]

print(df_summary.to_string(index=False))
print("="*100)

print(f"\n{'='*100}")
print(f"🏆 BEST ENSEMBLE TECHNIQUE: {best_ensemble['model_name']}")
print(f"{'='*100}")
print(f"  mAP:     {best_ensemble['mAP']:.2f}%")
print(f"  Rank-1:  {best_ensemble['rank1']:.2f}%")
print(f"  Rank-5:  {best_ensemble['rank5']:.2f}%")
print(f"  Rank-10: {best_ensemble['rank10']:.2f}%")
print(f"  Rank-20: {best_ensemble['rank20']:.2f}%")
print(f"{'='*100}")

avg_improvement_map = np.mean([i['mAP_imp'] for i in improvements])
avg_improvement_rank1 = np.mean([i['rank1_imp'] for i in improvements])
print(f"\n📊 AVERAGE ENSEMBLE IMPROVEMENT:")
print(f"  mAP:    {avg_improvement_map:+.2f}%")
print(f"  Rank-1: {avg_improvement_rank1:+.2f}%")
print(f"{'='*100}\n")

df_summary.to_csv('ensemble_comparison_summary.csv', index=False)
print("✓ Results saved to 'ensemble_comparison_summary.csv'")
print("✓ Visualization saved to 'ensemble_comprehensive_comparison.png'")

In [ ]:
def visualize_query_results(model, query_dataset, gallery_dataset, num_queries=5, top_k=10):
    """Visualize retrieval results for sample queries"""

    print(f"\n{'='*60}")
    print(f"EVALUATING {model_name.upper()}")
    print(f"{'='*60}")
    
    model.eval()
    start_time = time.time()
    
    def extract_features(dataset, desc):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        features, pids, camids = [], [], []
        
        with torch.no_grad():
            for img, pid, camid in tqdm(loader, desc=desc):
                img = img.to(device)
                output = model(img)
                
                feat = output[0] if isinstance(output, (tuple, list)) else output
                feat = F.normalize(feat, p=2, dim=1)
                
                features.append(feat.cpu())
                pids.extend(pid.tolist())
                camids.extend(camid.tolist())
        
        return torch.cat(features, 0), np.array(pids), np.array(camids)
    
    # Extract features
    q_feats, q_pids, q_camids = extract_features(query_dataset, f"{model_name} Query")
    g_feats, g_pids, g_camids = extract_features(gallery_dataset, f"{model_name} Gallery")
    
    extraction_time = time.time() - start_time
    
    # Compute distances
    print("Computing distance matrix...")
    m, n = q_feats.size(0), g_feats.size(0)
    dist_mat = torch.pow(q_feats, 2).sum(dim=1, keepdim=True).expand(m, n) + \
               torch.pow(g_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    dist_mat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    dist_mat = dist_mat.numpy()
    
    query_indices = np.random.choice(len(query_dataset), num_queries, replace=False)
    # dist_mat = results['dist_matrix']
    
    fig, axes = plt.subplots(num_queries, top_k + 1, figsize=(2*(top_k+1), 2*num_queries))
    
    for i, q_idx in enumerate(query_indices):
        # Get query image
        query_img, query_pid, _ = query_dataset[q_idx]
        
        # Get top-k gallery matches
        distances = dist_mat[q_idx]
        top_indices = np.argsort(distances)[:top_k]
        
        # Display query
        ax = axes[i, 0] if num_queries > 1 else axes[0]
        ax.imshow(query_img.permute(1, 2, 0) * 0.5 + 0.5)  # Denormalize
        ax.set_title(f'Query\nID: {query_pid.item()}', fontsize=9, fontweight='bold')
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), query_img.shape[2]-1, query_img.shape[1]-1,
                                   fill=False, edgecolor='blue', linewidth=3))
        
        # Display gallery matches
        for j, g_idx in enumerate(top_indices):
            gallery_img, gallery_pid, _ = gallery_dataset[g_idx]
            
            ax = axes[i, j+1] if num_queries > 1 else axes[j+1]
            ax.imshow(gallery_img.permute(1, 2, 0) * 0.5 + 0.5)
            
            is_correct = gallery_pid.item() == query_pid.item()
            color = 'green' if is_correct else 'red'
            label = '✓' if is_correct else '✗'
            
            ax.set_title(f'{label} Rank {j+1}\nID: {gallery_pid.item()}', 
                        fontsize=8, color=color, fontweight='bold')
            ax.axis('off')
            ax.add_patch(plt.Rectangle((0, 0), gallery_img.shape[2]-1, gallery_img.shape[1]-1,
                                      fill=False, edgecolor=color, linewidth=2))
    
    plt.tight_layout()
    plt.suptitle('Sample Query Results (Green=Correct, Red=Incorrect)', 
                fontsize=14, fontweight='bold', y=1.002)
    plt.show()



In [ ]:
import warnings
warnings.filterwarnings('ignore') # Hide all warnings
warnings.warn("This will not be shown")
print("Script continues without warning output")

In [ ]:
for model_name, model in trained_models.items():
    print(f"\nEvaluating {model_name}...")
    # results = evaluate_model(model, query_dataset, gallery_dataset, model_name)
    # evaluation_results.append(results)
    visualize_query_results(model, query_dataset, gallery_dataset, num_queries=5, top_k=5)

In [ ]:
plot_comparison(evaluation_results)
create_summary_table(evaluation_results)